In [57]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.inspection import permutation_importance

In [58]:
df = pd.read_csv(r'C:\Users\rsuru\Downloads\archive (2)\netflix_customer_churn.csv')
print(df.shape)                       # expect (5000, 14)
print(df.dtypes)
print(df.isna().sum().sum(), df.duplicated("customer_id").sum())
print(df["churned"].value_counts(normalize=True))
print(df.describe().T)

(5000, 14)
customer_id                   str
age                         int64
gender                        str
subscription_type             str
watch_hours               float64
last_login_days             int64
region                        str
device                        str
monthly_fee               float64
churned                     int64
payment_method                str
number_of_profiles          int64
avg_watch_time_per_day    float64
favorite_genre                str
dtype: object
0 0
churned
1    0.503
0    0.497
Name: proportion, dtype: float64
                         count      mean        std    min      25%    50%  \
age                     5000.0  43.84740  15.501128  18.00  30.0000  44.00   
watch_hours             5000.0  11.64945  12.014654   0.01   3.3375   8.00   
last_login_days         5000.0  30.08980  17.536078   0.00  15.0000  30.00   
monthly_fee             5000.0  13.68340   3.692062   8.99   8.9900  13.99   
churned                 5000.0   0.50300  

In [59]:
# Data-quality check: impossible values
bad = df[df["avg_watch_time_per_day"] > 24]
print(len(bad), "rows with >24 hrs watched per day")

10 rows with >24 hrs watched per day


In [60]:
df["mrr"] = df["monthly_fee"]
df["is_bad_watchtime"] = (df["avg_watch_time_per_day"] > 24).astype(int)
df["avg_watch_time_per_day"] = df["avg_watch_time_per_day"].clip(upper=24)
df["age_group"] = pd.cut(df["age"], [0,24,34,44,54,120],
                         labels=["<25","25-34","35-44","45-54","55+"])
df["recency_bucket"] = pd.cut(df["last_login_days"], [-1,7,14,30,10_000],
                              labels=["0-7","8-14","15-30","30+"])
df["engagement_band"] = pd.qcut(df["watch_hours"], 4,
                                labels=["Low","Medium","High","Power"])
df.to_csv(r'C:\Users\rsuru\OneDrive\Desktop\python\Netflix_customer\data\clean.csv', index=False)



In [61]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(ROOT / "data" / "clean.csv")

# Load environment variables
load_dotenv(ROOT / ".env", override=True)

# Columns that are IDs, the target, or derived from the target/other columns
drop_cols = ["customer_id", "churned", "mrr", "revenue_at_risk",
             "age_group", "recency_bucket", "engagement_band"]
X = df.drop(columns=drop_cols, errors="ignore") # ignores any that don't exist
y = df["churned"]

print("Features used:", X.columns.tolist())
print("Project root:", ROOT)
print("Dataset shape:", df.shape)

Features used: ['age', 'gender', 'subscription_type', 'watch_hours', 'last_login_days', 'region', 'device', 'monthly_fee', 'payment_method', 'number_of_profiles', 'avg_watch_time_per_day', 'favorite_genre', 'is_bad_watchtime']
Project root: c:\Users\rsuru\OneDrive\Desktop\python\Netflix_customer
Dataset shape: (5000, 19)


In [62]:
df_raw = pd.read_csv(r'C:\Users\rsuru\OneDrive\Desktop\python\Netflix_customer\data\netflix_customer_churn.csv')
df_raw.columns = df_raw.columns.str.strip().str.lower()
print(df_raw.shape)
df = df_raw.copy()

(5000, 14)


In [63]:
# Load environment variables
load_dotenv()

# Read database credentials
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    host=os.getenv("MYSQL_HOST"),
    port=int(os.getenv("MYSQL_PORT")),
    database=os.getenv("MYSQL_DATABASE")
)

engine = create_engine(connection_url)

# Load customer data
df = pd.read_sql("SELECT * FROM customers", engine)

# Convert DECIMAL columns into float
for c in ["watch_hours", "monthly_fee", "avg_watch_time_per_day", "mrr", "revenue_at_risk"]:
    if c in df.columns:
        df[c] = df[c].astype(float)

print(df.shape)       # expect (5000, 20)
print(df.dtypes)






(5000, 20)
customer_id                   str
age                         int64
gender                        str
subscription_type             str
watch_hours               float64
last_login_days             int64
region                        str
device                        str
monthly_fee               float64
churned                     int64
payment_method                str
number_of_profiles          int64
avg_watch_time_per_day    float64
favorite_genre                str
flag_watchtime_gt24        object
mrr                       float64
age_group                     str
recency_bucket                str
engagement_band               str
revenue_at_risk           float64
dtype: object


In [64]:
drop = ["customer_id", "churned", "mrr", "revenue_at_risk",
        "age_group", "recency_bucket", "engagement_band"]

def build(drop_cols):
    X = df.drop(columns=drop_cols, errors="ignore")
    cat = X.select_dtypes("object").columns.tolist()
    num = [c for c in X.columns if c not in cat]
    pre = ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), cat),
                             ("n", StandardScaler(), num)])
    return X, pre

models = {
    "logreg": LogisticRegression(max_iter=1000),
    "rf":     RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "hgb":    HistGradientBoostingClassifier(learning_rate=0.05, random_state=42),
}

y = df["churned"].astype(int)
X, pre = build(drop)
print("Features:", X.columns.tolist())

Xtr, Xte, ytr, yte = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
cv = StratifiedKFold(5, shuffle=True, random_state=42)

for name, m in models.items():
    pipe = Pipeline([("pre", pre), ("m", m)])
    print(name, "CV AUC:", cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="roc_auc").mean().round(3))

Features: ['age', 'gender', 'subscription_type', 'watch_hours', 'last_login_days', 'region', 'device', 'monthly_fee', 'payment_method', 'number_of_profiles', 'avg_watch_time_per_day', 'favorite_genre', 'flag_watchtime_gt24']
logreg CV AUC: 0.965


C:\Users\rsuru\AppData\Local\Temp\ipykernel_6940\1498942857.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = X.select_dtypes("object").columns.tolist()


rf CV AUC: 0.998
hgb CV AUC: 1.0


In [65]:
X2, pre2 = build(drop + ["avg_watch_time_per_day"])
X2tr, X2te, _, _ = train_test_split(X2, y, stratify=y, test_size=0.2, random_state=42)

for name in ["logreg", "hgb"]:
    pipe = Pipeline([("pre", pre2), ("m", models[name])])
    print(name, "CV AUC without avg_watch_time_per_day:",
          cross_val_score(pipe, X2tr, ytr, cv=cv, scoring="roc_auc").mean().round(3))

logreg CV AUC without avg_watch_time_per_day: 0.962


C:\Users\rsuru\AppData\Local\Temp\ipykernel_6940\1498942857.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = X.select_dtypes("object").columns.tolist()


hgb CV AUC without avg_watch_time_per_day: 0.999


In [66]:
best = Pipeline([("pre", pre), ("m", models["hgb"])]).fit(Xtr, ytr)
p = best.predict_proba(Xte)[:, 1]

print("Test AUC:", round(roc_auc_score(yte, p), 3))
print("PR-AUC:", round(average_precision_score(yte, p), 3))
print(classification_report(yte, p > 0.5))

Test AUC: 1.0
PR-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       497
           1       0.99      1.00      1.00       503

    accuracy                           0.99      1000
   macro avg       1.00      0.99      0.99      1000
weighted avg       1.00      0.99      0.99      1000



In [67]:


imp = permutation_importance(best, Xte, yte, scoring="roc_auc", n_repeats=10, random_state=42)
fi = pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False)
print(fi)

Path("../powerbi").mkdir(exist_ok=True)
fi.rename("importance").reset_index().rename(columns={"index": "feature"}) \
  .to_csv("../powerbi/feature_importance.csv", index=False)

avg_watch_time_per_day    0.178160
last_login_days           0.077967
number_of_profiles        0.046048
watch_hours               0.043040
payment_method            0.023044
subscription_type         0.011814
monthly_fee               0.000068
region                    0.000014
gender                    0.000004
favorite_genre            0.000003
flag_watchtime_gt24       0.000000
device                   -0.000001
age                      -0.000016
dtype: float64


In [68]:
offer_cost, save_rate = 3.0, 0.30        # assumptions: state them in your README
mrr_te = df.loc[Xte.index, "mrr"].values

best_t, best_val = 0, -1e9
for t in np.arange(0.05, 0.95, 0.05):
    target = p >= t
    value = (target * yte.values * mrr_te * save_rate * 12).sum() - target.sum() * offer_cost
    if value > best_val:
        best_t, best_val = t, value
print("Best threshold:", round(best_t, 2), "| Net value ($/yr):", round(best_val))

Best threshold: 0.15 | Net value ($/yr): 22442


In [69]:
# 1. Score every customer
df["churn_prob"] = best.predict_proba(X)[:, 1]
df["risk_tier"] = pd.cut(df["churn_prob"], [-0.01, 0.3, 0.6, 1.01],
                         labels=["Low", "Medium", "High"]).astype(str)
df["expected_loss"] = df["mrr"] * df["churn_prob"]

print(df["risk_tier"].value_counts())          # counts should add up to 5000

# 2. Save a CSV for Power BI
Path("../powerbi").mkdir(exist_ok=True)
df.to_csv("../powerbi/churn_scored.csv", index=False)

# 3. Save the scores to MySQL (reuses the `engine` from your first cell)
df[["customer_id", "churn_prob", "risk_tier", "expected_loss"]] \
    .to_sql("churn_scores", engine, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS n FROM churn_scores", engine))   # expect 5000

risk_tier
High      2505
Low       2481
Medium      14
Name: count, dtype: int64
      n
0  5000


In [70]:
# Create test prediction results
test_predictions = pd.DataFrame({
    "customer_id": df.loc[Xte.index, "customer_id"].values,
    "actual_churn": yte.values,
    "churn_probability": p,
    "predicted_churn": (p >= 0.50).astype(int)
})

# Save as CSV for Power BI
Path("../powerbi").mkdir(exist_ok=True)

test_predictions.to_csv(
    "../powerbi/test_predictions.csv",
    index=False
)

# Check the results
print(test_predictions.head())
print("Test prediction rows:", len(test_predictions))
print(test_predictions["actual_churn"].value_counts())
print(test_predictions["predicted_churn"].value_counts())

                            customer_id  actual_churn  churn_probability  \
0  61e49860-703a-4a67-82ad-5749094f5dc2             0           0.007934   
1  b281af12-1392-4e70-9dc8-b394fe8de6a8             1           0.998361   
2  3eaad6f4-7f15-4c49-87d7-e05105c2eea3             0           0.010350   
3  a02d6ab0-2fda-47eb-8656-158beb0200a4             1           0.995051   
4  b9802ea9-5c98-4d4a-a77c-6de878d22ea4             1           0.996188   

   predicted_churn  
0                0  
1                1  
2                0  
3                1  
4                1  
Test prediction rows: 1000
actual_churn
1    503
0    497
Name: count, dtype: int64
predicted_churn
1    504
0    496
Name: count, dtype: int64


In [71]:
cm = confusion_matrix(
    test_predictions["actual_churn"],
    test_predictions["predicted_churn"]
)

print(cm)

[[494   3]
 [  2 501]]


In [72]:
print("Accuracy:", accuracy_score(
    test_predictions["actual_churn"],
    test_predictions["predicted_churn"]
))

print("Precision:", precision_score(
    test_predictions["actual_churn"],
    test_predictions["predicted_churn"]
))

print("Recall:", recall_score(
    test_predictions["actual_churn"],
    test_predictions["predicted_churn"]
))

print("F1 Score:", f1_score(
    test_predictions["actual_churn"],
    test_predictions["predicted_churn"]
))

Accuracy: 0.995
Precision: 0.9940476190476191
Recall: 0.9960238568588469
F1 Score: 0.9950347567030785


In [73]:
# Data quality validation

print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate customer IDs:")
print(df["customer_id"].duplicated().sum())

print("\nChurn distribution:")
print(df["churned"].value_counts())

print("\nInvalid watch time:")
print((df["avg_watch_time_per_day"] > 24).sum())

Dataset shape: (5000, 23)

Missing values:
customer_id                  0
age                          0
gender                       0
subscription_type            0
watch_hours                  0
last_login_days              0
region                       0
device                       0
monthly_fee                  0
churned                      0
payment_method               0
number_of_profiles           0
avg_watch_time_per_day       0
favorite_genre               0
flag_watchtime_gt24       5000
mrr                          0
age_group                    0
recency_bucket               0
engagement_band              0
revenue_at_risk           5000
churn_prob                   0
risk_tier                    0
expected_loss                0
dtype: int64

Duplicate customer IDs:
0

Churn distribution:
churned
1    2515
0    2485
Name: count, dtype: int64

Invalid watch time:
0
